# fastgws

> Python client for Google Workspace and Google APIs built from discovery docs

fastgws builds async Python clients for Google APIs from Google's discovery documents. Each service gets a small Python surface: resource groups become attributes, operations become awaitable methods, and responses come back as lightweight objects instead of raw JSON dictionaries.

## Installation

Install from [pypi][pypi]:

```sh
$ pip install fastgws
```

Or install the latest development version from [GitHub][repo]:

```sh
$ pip install git+https://github.com/answerdotai/fastgws.git
```

[repo]: https://github.com/answerdotai/fastgws
[docs]: https://answerdotai.github.io/fastgws/
[pypi]: https://pypi.org/project/fastgws/

## How to use

Import the service clients you want to use. Each client is built from Google's discovery documents and exposes resource groups as Python attributes, so calls look like `await drive.files.list(...)` or `await calendar.events.list(...)`.

In [ ]:
from fastgws import Calendar, Docs, Drive, GMail, Places
from fastgws.auth import *

fastgws supports both OAuth credentials and API keys. OAuth is the usual choice for Google Workspace APIs such as Gmail, Calendar, Drive, and Docs, because these APIs act on behalf of a user and require explicit scopes.

`oauth_creds` looks for a Google OAuth client file named `credentials.json` in the fastgws config directory, which defaults to `~/.config/fastgws/credentials.json`. Create this OAuth client in Google Cloud Console as a web application, then add the redirect URI used below (`https://oauth.appapis.org/redirect`) to the client's authorized redirect URIs.

Pass `oauth_creds` the scopes your app needs. On the first run it prints or displays an authorization link; visit that link, approve access, then paste the returned code back into the prompt. fastgws saves the resulting token in the same config directory and reuses it on later runs when the saved token covers the requested scopes, including when you request only a subset of the scopes already granted.

API keys are useful for public Google APIs that support key-based access, such as Places. You can pass `api_key=...` directly or set `GOOGLE_API_KEY` or `GWS_API_KEY` in the environment.

In [ ]:
creds = oauth_creds(scopes=['https://www.googleapis.com/auth/calendar',
                            'https://www.googleapis.com/auth/documents',
                            'https://www.googleapis.com/auth/drive.readonly',
                            'https://www.googleapis.com/auth/gmail.readonly'],
                    redirect_uri='https://oauth.appapis.org/redirect')

HTML(<b>Auth complete</b>)

Responses are converted into lightweight Python objects. Known Google resource kinds get more specific classes such as `FileList`, `Events`, or `Event`; when a service does not provide enough schema information, fastgws falls back to the base `GWSObject`. Either way, fields are available as attributes as well as dictionary keys.

Use Docs to create a document, apply batch updates, and read the document back. The API accepts the same request dictionaries documented by Google, while fastgws handles auth, transport, and object conversion.

In [ ]:
docs = Docs(creds=creds)
doc = await docs.documents.create(title='fastgws test doc')
doc

<div class="prose" markdown="1">

```python
GWSObject(title='fastgws test doc', documentId='1ohlMW4OttiYNExpTR4SfVdDpGwW7qvyJAYY-YcxaYW0', body=1, documentStyle=11, namedStyles=1, tabs=1)
```

</div>

In [ ]:
await docs.documents.batch_update(document_id=doc.documentId,
                                  requests=[{'insertText': {'location': {'index': 1},
                                             'text': 'Hello from fastgws\n'}}])

<div class="prose" markdown="1">

```python
GWSObject(documentId='1ohlMW4OttiYNExpTR4SfVdDpGwW7qvyJAYY-YcxaYW0', replies=1, writeControl=1)
```

</div>

In [ ]:
def doc_text(doc):
    return ''.join(e.textRun.content for b in doc.body.content if 'paragraph' in b for e in b.paragraph.elements if 'textRun' in e)

doc = await docs.documents.get(document_id=doc.documentId)
txt = doc_text(doc)
print(txt)

Hello from fastgws




Use Drive to search files and inspect metadata. This example returns a `FileList`, and its `files` collection contains file objects with attributes such as `id`, `name`, and `mimeType`.

In [ ]:
drive = Drive(creds=creds)
fs = await drive.files.list(q="name contains 'fastgws' and trashed=false", page_size=10)
fs, fs.files[0]

(FileList(kind='drive#fileList', files=2),
 File(id='1ohlMW4OttiYNExpTR4SfVdDpGwW7qvyJAYY-YcxaYW0', name='fastgws test doc', mimeType='application/vnd.google-apps.document', kind='drive#file'))

Use Gmail to search messages with the Gmail query syntax. The result is still a Python object, so you can inspect message ids, thread ids, and any fields returned by the API without digging through raw JSON first.

In [ ]:
gmail = GMail(creds=creds)
msgs = await gmail.users.messages.list(user_id='me', max_results=10)
msgs

<div class="prose" markdown="1">

```python
GWSObject(messages=10)
```

</div>

Use Calendar to create, update, delete, and search events.

In [ ]:
calendar = Calendar(creds=creds)
event = await calendar.events.insert(calendar_id='primary',
                                    summary='fastgws test event',
                                    start={'dateTime': '2030-01-01T09:00:00Z'},
                                    end={'dateTime': '2030-01-01T09:30:00Z'})
event

<div class="prose" markdown="1">

```python
Event(id='t997kgs5meii3h0e2ft6g66omo', summary='fastgws test event', kind='calendar#event', creator=2, organizer=2, start=2, end=2, reminders=1)
```

</div>

In [ ]:
events = await calendar.events.list(calendar_id='primary', q='fastgws test event',
                                    max_results=10, single_events=True, order_by='startTime')
events, events['items'][0]

(Events(summary='nc@answer.ai', kind='calendar#events', defaultReminders=1, items=1),
 Event(id='t997kgs5meii3h0e2ft6g66omo', summary='fastgws test event', kind='calendar#event', creator=2, organizer=2, start=2, end=2, reminders=1))

In [ ]:
await calendar.events.delete(calendar_id='primary', event_id=event.id)

''

Use API-key services the same way. Places can run with an API key instead of OAuth credentials, and this example asks Google to return only the fields needed to render a link.

In [ ]:
from IPython.display import Markdown

places = Places()
res = await places.places.search_text(text_query='coffee near San Francisco',
                                      _headers={'X-Goog-FieldMask':'places.displayName,places.formattedAddress,places.location,places.googleMapsUri'})
p = res.places[0]
Markdown(f'[{p.displayName.text}]({p.googleMapsUri})')

<div class="prose" markdown="1">

[787 Coffee](https://maps.google.com/?cid=978151600972640730&g_mp=Cidnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJjaFRleHQQAhgEIAA)

</div>

fastgws can also create service clients dynamically from Google's discovery index. If Google publishes a discovery document for a service, you can usually import that service by name, for example `from fastgws import Sheets`, then use it with the same `creds`, `token`, or `api_key` arguments shown above.